# Tutorial 4: Clustering and Dimensionality Reduction

Welcome to the fourth tutorial in our statistical learning series! In
this notebook, we’ll explore unsupervised learning techniques, focusing
on clustering and dimensionality reduction - powerful methods for
discovering patterns and structure in unlabeled data.

## Learning Objectives

By the end of this tutorial, you’ll be able to: - Apply various
clustering algorithms to group similar data points - Determine the
optimal number of clusters using quantitative methods - Reduce data
dimensionality while preserving important information - Visualize
high-dimensional data in lower-dimensional spaces - Interpret clustering
results and principal components

## 1. Setup and Introduction

Let’s begin by importing the necessary libraries.

``` python
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

# Import our clustering utilities
from statistics_lessons.ml_models.clustering import (
    kmeans_clustering,
    hierarchical_clustering,
    dbscan_clustering,
    elbow_inertia,
    silhouette_analysis
)
from statistics_lessons.projects.data_loaders import load_iris_data, load_wine_data

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
```

## 2. Clustering: Finding Natural Groupings in Data

Clustering is an unsupervised learning technique that identifies natural
groupings in data by maximizing within-group similarity and
between-group differences.

### 2.1 Generating Synthetic Clusters

Let’s start by creating a synthetic dataset with clearly defined
clusters to understand the basic concepts.

``` python
# Set random seed for reproducibility
np.random.seed(42)

# Generate three clusters
n_samples = 150
n_features = 2
n_clusters = 3

# Create cluster centers
centers = [(-5, -5), (0, 5), (5, -2)]

# Generate data points around each center
X = np.vstack([
    np.random.randn(n_samples // 3, n_features) + centers[0],
    np.random.randn(n_samples // 3, n_features) + centers[1],
    np.random.randn(n_samples // 3, n_features) + centers[2]
])

# Shuffle the data
np.random.shuffle(X)

# Visualize the data
plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], alpha=0.7)
plt.title('Synthetic Clustered Data')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()
```

### 2.2 K-Means Clustering

K-means is one of the most popular clustering algorithms. It partitions
data into k clusters by minimizing the within-cluster variance.

``` python
# Apply K-means clustering
model, labels = kmeans_clustering(X, n_clusters=3, random_state=42)

# Plot the clusters
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', alpha=0.7)
plt.scatter(model.cluster_centers_[:, 0], model.cluster_centers_[:, 1], 
            marker='X', s=200, c='red', label='Centroids')
plt.title('K-means Clustering (k=3)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True)
plt.show()

# Show cluster information
cluster_sizes = pd.Series(labels).value_counts().sort_index()
print("Cluster sizes:")
for i, size in enumerate(cluster_sizes):
    print(f"Cluster {i}: {size} points")
```

### 2.3 Determining the Optimal Number of Clusters

In real-world scenarios, the true number of clusters is often unknown.
Let’s explore methods to determine the optimal number of clusters.

#### The Elbow Method

The elbow method looks at the within-cluster sum of squares (inertia) as
a function of the number of clusters.

``` python
# Compute inertia for different numbers of clusters
k_range = range(1, 11)
inertias = elbow_inertia(X, k_range)

# Plot the elbow curve
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), list(inertias.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Optimal k')
plt.grid(True)
plt.show()
```

#### Silhouette Analysis

The silhouette score measures how similar points are to their own
cluster compared to other clusters.

``` python
# Compute silhouette scores for different numbers of clusters
k_range = range(2, 11)  # Silhouette score needs at least 2 clusters
silhouette_scores = silhouette_analysis(X, k_range)

# Plot silhouette scores
plt.figure(figsize=(10, 6))
plt.plot(list(k_range), list(silhouette_scores.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Optimal k')
plt.grid(True)
plt.show()
```

### 2.4 Hierarchical Clustering

Hierarchical clustering builds a tree of clusters by either merging
(agglomerative) or splitting (divisive) groups.

``` python
# Apply hierarchical clustering
model_hier, labels_hier = hierarchical_clustering(X, n_clusters=3, linkage='ward')

# Plot the clusters
plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels_hier, cmap='viridis', alpha=0.7)
plt.title('Hierarchical Clustering (k=3, Ward linkage)')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.show()

# Create dendrogram
from scipy.cluster.hierarchy import dendrogram, linkage

# Use a sample of points for the dendrogram to avoid clutter
sample_size = min(100, X.shape[0])
sample_indices = np.random.choice(range(X.shape[0]), sample_size, replace=False)
X_sample = X[sample_indices]

# Compute linkage matrix
linked = linkage(X_sample, method='ward')

plt.figure(figsize=(12, 7))
dendrogram(linked, orientation='top')
plt.title('Hierarchical Clustering Dendrogram (Ward linkage)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.tight_layout()
plt.show()
```

### 2.5 DBSCAN Clustering

DBSCAN (Density-Based Spatial Clustering of Applications with Noise)
identifies clusters as dense regions separated by sparser regions. It
can discover clusters of arbitrary shape and doesn’t require specifying
the number of clusters in advance.

``` python
# Apply DBSCAN clustering
model_dbscan, labels_dbscan = dbscan_clustering(X, eps=1.5, min_samples=5)

# Number of clusters (excluding noise points labeled as -1)
n_clusters_dbscan = len(set(labels_dbscan)) - (1 if -1 in labels_dbscan else 0)
print(f"DBSCAN found {n_clusters_dbscan} clusters and {np.sum(labels_dbscan == -1)} noise points")

# Plot the clusters
plt.figure(figsize=(10, 6))
unique_labels = set(labels_dbscan)
colors = plt.cm.viridis(np.linspace(0, 1, len(unique_labels)))

for k, col in zip(unique_labels, colors):
    if k == -1:
        # Black used for noise
        col = [0, 0, 0, 1]

    class_member_mask = (labels_dbscan == k)
    xy = X[class_member_mask]
    plt.scatter(xy[:, 0], xy[:, 1], s=50, c=[col], alpha=0.7,
                label='Cluster' if k != -1 else 'Noise')

plt.title('DBSCAN Clustering')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.legend()
plt.show()
```

### Interactive Exercise 1: Comparing Clustering Algorithms

🔍 **Compare the three clustering methods and answer these questions:**

1.  How do the cluster assignments differ between K-means, hierarchical
    clustering, and DBSCAN?
2.  Which method best captures the true structure of the synthetic data?
3.  What are the advantages and limitations of each algorithm?
4.  How did the clustering parameters (like k or eps) affect the
    results?

<details>
<summary>
Click for answers
</summary>

1.  Differences in cluster assignments:
    -   K-means tends to create spherical, equally-sized clusters
    -   Hierarchical clustering can create more flexible cluster shapes
        depending on the linkage
    -   DBSCAN can identify irregularly shaped clusters and marks
        outliers as noise
2.  For this synthetic dataset with well-separated, roughly spherical
    clusters, all methods should perform reasonably well. However:
    -   K-means and hierarchical clustering should produce very similar
        results since the data was generated from Gaussian distributions
    -   DBSCAN might identify slightly different boundaries, especially
        if there are outliers
3.  Advantages and limitations:
    -   K-means:
        -   Advantages: Simple, fast, works well for spherical clusters
        -   Limitations: Requires specifying k, sensitive to outliers,
            works poorly for non-spherical clusters
    -   Hierarchical clustering:
        -   Advantages: Produces a dendrogram showing relationships,
            doesn’t require pre-specifying clusters
        -   Limitations: Computationally expensive for large datasets,
            different linkage methods can produce different results
    -   DBSCAN:
        -   Advantages: Doesn’t require specifying number of clusters,
            can find arbitrarily shaped clusters, identifies outliers
        -   Limitations: Sensitive to parameter choices (eps,
            min_samples), struggles with clusters of varying densities
4.  Parameter effects:
    -   K-means: Increasing k splits clusters further, potentially
        dividing natural groupings
    -   Hierarchical clustering: Different linkage methods (ward,
        complete, average, etc.) can yield different hierarchies
    -   DBSCAN: Larger eps values create fewer, more inclusive clusters;
        smaller values create more, tighter clusters

</details>

## 3. Clustering Real Data: Iris Dataset

Let’s apply clustering to the classic Iris dataset and see how well our
algorithms recover the known flower species.

``` python
# Load the Iris dataset
X_iris, y_iris = load_iris_data()

# Scale the features
scaler = StandardScaler()
X_iris_scaled = scaler.fit_transform(X_iris)

# Apply K-means clustering
model_iris, labels_iris = kmeans_clustering(X_iris_scaled, n_clusters=3, random_state=42)

# Create a DataFrame with original data, true labels, and cluster assignments
iris_results = pd.DataFrame(X_iris.values, columns=X_iris.columns)
iris_results['Species'] = y_iris
iris_results['Cluster'] = labels_iris

# Evaluate clustering against true labels
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

ari = adjusted_rand_score(y_iris, labels_iris)
nmi = normalized_mutual_info_score(y_iris, labels_iris)

print(f"Clustering evaluation metrics:")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

# Create a cross-tabulation of species vs. clusters
ct = pd.crosstab(iris_results['Species'], iris_results['Cluster'], 
                 rownames=['Species'], colnames=['Cluster'])
print("\nCross-tabulation of species vs. clusters:")
print(ct)

# Visualize the clusters using the first two features
plt.figure(figsize=(12, 5))

# Plot with true labels
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_iris.iloc[:, 0], X_iris.iloc[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('Iris Dataset: True Species')
plt.xlabel(X_iris.columns[0])
plt.ylabel(X_iris.columns[1])
plt.legend(handles=scatter.legend_elements()[0], labels=y_iris.unique())

# Plot with cluster assignments
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_iris.iloc[:, 0], X_iris.iloc[:, 1], c=labels_iris, cmap='viridis', alpha=0.7)
plt.title('Iris Dataset: K-means Clusters')
plt.xlabel(X_iris.columns[0])
plt.ylabel(X_iris.columns[1])
plt.legend(handles=scatter.legend_elements()[0], labels=np.unique(labels_iris))

plt.tight_layout()
plt.show()
```

### 3.1 Exploring the Optimal Number of Clusters for Iris

``` python
# Elbow method for Iris
k_range = range(1, 11)
inertias_iris = elbow_inertia(X_iris_scaled, k_range)

# Silhouette analysis for Iris
k_range_sil = range(2, 11)
silhouette_scores_iris = silhouette_analysis(X_iris_scaled, k_range_sil)

# Plot both methods
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(list(k_range), list(inertias_iris.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Iris Dataset')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(list(k_range_sil), list(silhouette_scores_iris.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Iris Dataset')
plt.grid(True)

plt.tight_layout()
plt.show()
```

## 4. Dimensionality Reduction: PCA

Principal Component Analysis (PCA) is a technique for reducing the
dimensionality of data while preserving as much variance as possible.

### 4.1 Understanding PCA with a Synthetic Example

``` python
# Generate correlated data
np.random.seed(42)
n_samples = 200

# Create a covariance matrix for correlated features
cov = np.array([[3, 2.5], [2.5, 2]])
mean = [0, 0]

# Generate data from a multivariate normal distribution
X_corr = np.random.multivariate_normal(mean, cov, n_samples)

# Apply PCA
pca = PCA()
X_pca = pca.fit_transform(X_corr)

# Plot original data with principal components
plt.figure(figsize=(10, 6))
plt.scatter(X_corr[:, 0], X_corr[:, 1], alpha=0.7)

# Plot principal components
origin = [0, 0]
plt.arrow(origin[0], origin[1], pca.components_[0, 0] * 3, pca.components_[0, 1] * 3,
          head_width=0.2, head_length=0.3, fc='red', ec='red', label='First PC')
plt.arrow(origin[0], origin[1], pca.components_[1, 0] * 3, pca.components_[1, 1] * 3,
          head_width=0.2, head_length=0.3, fc='green', ec='green', label='Second PC')

plt.title('Data with Principal Components')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.grid(True)
plt.axis('equal')
plt.legend()
plt.show()

# Show explained variance
print(f"Explained variance ratio: {pca.explained_variance_ratio_}")
print(f"Total explained variance: {np.sum(pca.explained_variance_ratio_):.4f}")

# Plot explained variance
plt.figure(figsize=(8, 5))
plt.bar(range(1, len(pca.explained_variance_ratio_) + 1), pca.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Component')
plt.xticks(range(1, len(pca.explained_variance_ratio_) + 1))
plt.grid(True)
plt.show()
```

### 4.2 PCA on the Iris Dataset

``` python
# Apply PCA to the Iris dataset
pca_iris = PCA()
X_iris_pca = pca_iris.fit_transform(X_iris_scaled)

# Plot explained variance
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca_iris.explained_variance_ratio_) + 1), 
        pca_iris.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Component')
plt.xticks(range(1, len(pca_iris.explained_variance_ratio_) + 1))
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(pca_iris.explained_variance_ratio_) + 1),
         np.cumsum(pca_iris.explained_variance_ratio_), 'o-')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualize the data in the first two principal components
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('Iris Dataset in PCA Space')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True)
plt.legend(handles=scatter.legend_elements()[0], labels=y_iris.unique())
plt.show()

# Feature loadings (how each original feature contributes to each principal component)
loadings = pd.DataFrame(pca_iris.components_.T, columns=[f'PC{i+1}' for i in range(pca_iris.components_.shape[0])],
                      index=X_iris.columns)
print("PCA Loadings (Feature Contributions to Principal Components):")
print(loadings)

# Visualize feature loadings
plt.figure(figsize=(10, 6))
sns.heatmap(loadings, annot=True, cmap='coolwarm', center=0)
plt.title('Feature Loadings for Principal Components')
plt.tight_layout()
plt.show()
```

### 4.3 3D Visualization with PCA

``` python
# If we have at least 3 principal components, visualize in 3D
if X_iris_pca.shape[1] >= 3:
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    scatter = ax.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], X_iris_pca[:, 2],
                         c=y_iris, cmap='viridis', alpha=0.7)
    
    ax.set_title('Iris Dataset in 3D PCA Space')
    ax.set_xlabel('First Principal Component')
    ax.set_ylabel('Second Principal Component')
    ax.set_zlabel('Third Principal Component')
    
    plt.legend(handles=scatter.legend_elements()[0], labels=y_iris.unique())
    plt.show()
```

### Interactive Exercise 2: Interpreting PCA Results

🔍 **Analyze the PCA results for the Iris dataset and answer these
questions:**

1.  How many principal components would you retain based on the
    explained variance? Why?
2.  What do the feature loadings tell you about the meaning of each
    principal component?
3.  How well does the 2D PCA visualization separate the three Iris
    species?
4.  How might you use PCA results to simplify the original dataset?

<details>
<summary>
Click for answers
</summary>

1.  Choosing the number of principal components:
    -   Based on the explained variance plot, the first two components
        typically explain 95-98% of the variance in the Iris dataset
    -   This suggests that retaining just 2 components would be
        sufficient for most analyses
    -   If we need even higher accuracy, we might keep 3 components
        which should capture nearly all of the variance
2.  Interpreting feature loadings:
    -   The first principal component (PC1) usually has high loadings
        for petal length and petal width, indicating it primarily
        represents petal size
    -   The second principal component (PC2) often has higher loadings
        for sepal width (often negative) and sepal length, suggesting it
        captures sepal shape variations
    -   This means PC1 primarily distinguishes Iris setosa from the
        other species (which have larger petals), while PC2 helps
        separate Iris versicolor from Iris virginica
3.  Species separation in 2D PCA:
    -   Iris setosa typically forms a very distinct cluster in PCA space
    -   Iris versicolor and Iris virginica show some overlap but are
        reasonably well separated
    -   This mirrors the actual biological relationships, as setosa is
        more distantly related to the other two species
4.  Using PCA to simplify the dataset:
    -   We could replace the original 4 features with just 2 principal
        components for visualization
    -   For modeling, we could use the reduced dataset as input
        features, potentially improving performance by removing noise
    -   The PCA transformation could be applied to new iris specimens to
        classify them using less measurements
    -   We might design a simplified measurement process that focuses on
        the features with the highest loadings

</details>

## 5. Combining Clustering with PCA

PCA and clustering are often used together, with PCA reducing dimensions
before applying clustering algorithms.

``` python
# Cluster the Iris dataset in PCA space
X_iris_pca_2d = X_iris_pca[:, :2]  # Use just the first two principal components
model_pca, labels_pca = kmeans_clustering(X_iris_pca_2d, n_clusters=3, random_state=42)

# Evaluate clustering in PCA space
ari_pca = adjusted_rand_score(y_iris, labels_pca)
nmi_pca = normalized_mutual_info_score(y_iris, labels_pca)

print(f"Clustering in PCA space:")
print(f"Adjusted Rand Index: {ari_pca:.4f}")
print(f"Normalized Mutual Information: {nmi_pca:.4f}")

# Compare with clustering in original space
print(f"\nClustering in original space:")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

# Visualize clusters in PCA space
plt.figure(figsize=(12, 5))

# True species
plt.subplot(1, 2, 1)
scatter = plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('PCA Space: True Species')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True)
plt.legend(handles=scatter.legend_elements()[0], labels=y_iris.unique())

# Cluster assignments
plt.subplot(1, 2, 2)
scatter = plt.scatter(X_iris_pca[:, 0], X_iris_pca[:, 1], c=labels_pca, cmap='viridis', alpha=0.7)
centers = model_pca.cluster_centers_
plt.scatter(centers[:, 0], centers[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.title('PCA Space: K-means Clusters')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True)
plt.legend()

plt.tight_layout()
plt.show()

# Create a cross-tabulation of species vs. PCA-based clusters
ct_pca = pd.crosstab(y_iris, labels_pca, rownames=['Species'], colnames=['Cluster'])
print("\nCross-tabulation of species vs. PCA-based clusters:")
print(ct_pca)
```

## 6. Dimensionality Reduction with t-SNE

t-SNE (t-Distributed Stochastic Neighbor Embedding) is another popular
dimensionality reduction technique that’s particularly good at
preserving local structure.

``` python
from sklearn.manifold import TSNE

# Apply t-SNE to the Iris dataset
tsne = TSNE(n_components=2, random_state=42)
X_iris_tsne = tsne.fit_transform(X_iris_scaled)

# Visualize the results
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_iris_tsne[:, 0], X_iris_tsne[:, 1], c=y_iris, cmap='viridis', alpha=0.7)
plt.title('Iris Dataset: t-SNE Visualization')
plt.xlabel('t-SNE Dimension 1')
plt.ylabel('t-SNE Dimension 2')
plt.grid(True)
plt.legend(handles=scatter.legend_elements()[0], labels=y_iris.unique())
plt.show()

# Cluster in t-SNE space
model_tsne, labels_tsne = kmeans_clustering(X_iris_tsne, n_clusters=3, random_state=42)

# Evaluate clustering in t-SNE space
ari_tsne = adjusted_rand_score(y_iris, labels_tsne)
nmi_tsne = normalized_mutual_info_score(y_iris, labels_tsne)

print(f"Clustering in t-SNE space:")
print(f"Adjusted Rand Index: {ari_tsne:.4f}")
print(f"Normalized Mutual Information: {nmi_tsne:.4f}")

# Compare with previous results
print(f"\nClustering in original space:")
print(f"Adjusted Rand Index: {ari:.4f}")
print(f"Normalized Mutual Information: {nmi:.4f}")

print(f"\nClustering in PCA space:")
print(f"Adjusted Rand Index: {ari_pca:.4f}")
print(f"Normalized Mutual Information: {nmi_pca:.4f}")
```

### Interactive Exercise 3: Comparing Dimensionality Reduction Techniques

🔍 **Compare PCA and t-SNE visualizations and answer these questions:**

1.  How do the PCA and t-SNE visualizations of the Iris dataset differ?
2.  Which method better separates the three Iris species visually? Why?
3.  How do clustering results compare when applied to the original data,
    PCA-reduced data, and t-SNE-reduced data?
4.  What are the advantages and disadvantages of each dimensionality
    reduction technique?

<details>
<summary>
Click for answers
</summary>

1.  Differences between PCA and t-SNE visualizations:
    -   PCA creates a linear projection that maximizes variance
    -   t-SNE creates a non-linear embedding that preserves local
        neighborhood structure
    -   PCA shows global structure with straight-line distances having
        meaning
    -   t-SNE often creates more distinct clusters but distances between
        clusters may not be meaningful
2.  Better visual separation:
    -   t-SNE typically provides better visual separation of the Iris
        species
    -   This is because t-SNE focuses on maintaining local similarities,
        which helps separate distinct groups
    -   PCA is constrained by linearity, so it may not fully separate
        species that aren’t linearly separable
    -   However, t-SNE’s focus on local structure means it could
        sometimes exaggerate small differences
3.  Clustering comparison:
    -   Original data: Clustering directly on the 4 features usually
        performs well
    -   PCA-reduced data: Clustering on 2 principal components may
        perform slightly worse but is often very close
    -   t-SNE-reduced data: Often produces the best visual clustering,
        but can sometimes overfit to the data structure
    -   The adjusted Rand index and NMI scores quantify how well each
        approach recovers the true species
4.  Advantages and disadvantages:
    -   PCA:
        -   Advantages: Fast, interpretable (loadings show feature
            importance), preserves global structure
        -   Disadvantages: Limited to linear projections, may miss
            complex non-linear patterns
    -   t-SNE:
        -   Advantages: Excellent for visualization, preserves local
            structure, can reveal complex patterns
        -   Disadvantages: Computationally intensive, non-deterministic,
            difficult to interpret, doesn’t preserve global structure

</details>

## 7. Advanced Applications: Wine Dataset

Let’s apply what we’ve learned to a more complex dataset: the Wine
dataset.

``` python
# Load the Wine dataset
X_wine, y_wine = load_wine_data()

# Scale the features
scaler_wine = StandardScaler()
X_wine_scaled = scaler_wine.fit_transform(X_wine)

# Apply PCA
pca_wine = PCA()
X_wine_pca = pca_wine.fit_transform(X_wine_scaled)

# Plot explained variance
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.bar(range(1, len(pca_wine.explained_variance_ratio_) + 1), 
        pca_wine.explained_variance_ratio_)
plt.xlabel('Principal Component')
plt.ylabel('Explained Variance Ratio')
plt.title('Explained Variance by Component')
plt.xticks(range(1, len(pca_wine.explained_variance_ratio_) + 1))
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(range(1, len(pca_wine.explained_variance_ratio_) + 1),
         np.cumsum(pca_wine.explained_variance_ratio_), 'o-')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('Cumulative Explained Variance')
plt.grid(True)

plt.tight_layout()
plt.show()

# Visualize the first two principal components
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X_wine_pca[:, 0], X_wine_pca[:, 1], c=y_wine, cmap='viridis', alpha=0.7)
plt.title('Wine Dataset in PCA Space')
plt.xlabel('First Principal Component')
plt.ylabel('Second Principal Component')
plt.grid(True)
plt.legend(handles=scatter.legend_elements()[0], labels=np.unique(y_wine))
plt.show()

# Apply K-means clustering to the PCA-reduced data
n_clusters_wine = len(np.unique(y_wine))
model_wine, labels_wine = kmeans_clustering(X_wine_pca[:, :2], n_clusters=n_clusters_wine, random_state=42)

# Evaluate clustering results
ari_wine = adjusted_rand_score(y_wine, labels_wine)
nmi_wine = normalized_mutual_info_score(y_wine, labels_wine)

print(f"Wine clustering evaluation:")
print(f"Adjusted Rand Index: {ari_wine:.4f}")
print(f"Normalized Mutual Information: {nmi_wine:.4f}")

# Create a cross-tabulation of wine class vs. clusters
ct_wine = pd.crosstab(y_wine, labels_wine, rownames=['Wine Class'], colnames=['Cluster'])
print("\nCross-tabulation of wine class vs. clusters:")
print(ct_wine)

# Feature loadings
loadings_wine = pd.DataFrame(pca_wine.components_.T[:, :2], 
                          columns=['PC1', 'PC2'],
                          index=X_wine.columns)

# Visualize feature loadings
plt.figure(figsize=(12, 8))
plt.barh(loadings_wine.index, loadings_wine['PC1'], alpha=0.7, label='PC1')
plt.barh(loadings_wine.index, loadings_wine['PC2'], alpha=0.7, label='PC2')
plt.xlabel('Loading Value')
plt.ylabel('Feature')
plt.title('Feature Loadings for First Two Principal Components')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()
```

## 8. Practice Exercise: Clustering and Visualizing Customer Segments

Now it’s your turn to apply clustering and dimensionality reduction to a
customer segmentation problem.

``` python
# Load the customer dataset
from sklearn.datasets import make_blobs

# Generate synthetic customer data
X_customers, _ = make_blobs(n_samples=500, centers=4, random_state=42, cluster_std=[1.0, 2.0, 0.5, 1.5])

# Add feature names
customer_data = pd.DataFrame(X_customers, columns=['Annual Income', 'Spending Score'])

# Standardize the features
scaler_customers = StandardScaler()
X_customers_scaled = scaler_customers.fit_transform(customer_data)

# Your task:
# 1. Determine the optimal number of customer segments using the elbow method and silhouette analysis
# 2. Apply K-means clustering with the optimal number of clusters
# 3. Visualize the customer segments
# 4. Interpret the characteristics of each segment
```

<details>
<summary>
Click for sample solution
</summary>

``` python
# 1. Determine the optimal number of clusters
k_range = range(1, 11)
inertias_customers = elbow_inertia(X_customers_scaled, k_range)

k_range_sil = range(2, 11)
silhouette_scores_customers = silhouette_analysis(X_customers_scaled, k_range_sil)

# Plot both methods
plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(list(k_range), list(inertias_customers.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Method for Customer Segmentation')
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(list(k_range_sil), list(silhouette_scores_customers.values()), 'o-')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Analysis for Customer Segmentation')
plt.grid(True)

plt.tight_layout()
plt.show()

# Based on the plots, let's choose k=4
optimal_k = 4

# 2. Apply K-means clustering
model_customers, labels_customers = kmeans_clustering(X_customers_scaled, n_clusters=optimal_k, random_state=42)

# Add cluster labels to the original data
customer_data['Cluster'] = labels_customers

# 3. Visualize the customer segments
plt.figure(figsize=(10, 6))
colors = ['red', 'green', 'blue', 'purple', 'orange', 'brown', 'pink', 'gray', 'olive', 'cyan']

for i in range(optimal_k):
    cluster_data = customer_data[customer_data['Cluster'] == i]
    plt.scatter(cluster_data['Annual Income'], cluster_data['Spending Score'], 
                color=colors[i], alpha=0.7, label=f'Cluster {i}')

# Plot cluster centers
centers = scaler_customers.inverse_transform(model_customers.cluster_centers_)
plt.scatter(centers[:, 0], centers[:, 1], s=200, marker='X', c='black', label='Centroids')

plt.title('Customer Segments')
plt.xlabel('Annual Income')
plt.ylabel('Spending Score')
plt.legend()
plt.grid(True)
plt.show()

# 4. Interpret the characteristics of each segment
cluster_summary = customer_data.groupby('Cluster').mean()
print("Cluster Characteristics:")
print(cluster_summary)

# Add count of customers in each segment
cluster_counts = customer_data['Cluster'].value_counts().sort_index()
cluster_summary['Count'] = cluster_counts
cluster_summary['Percentage'] = 100 * cluster_counts / len(customer_data)

print("\nCluster Summary:")
print(cluster_summary)

# Interpretation:
# Cluster 0: High income, high spending - "Premium Customers"
# Cluster 1: Low income, low spending - "Budget Conscious"
# Cluster 2: High income, low spending - "Potential Targets"
# Cluster 3: Low income, high spending - "Occasional Splurgers"

# Visualization of segment characteristics
plt.figure(figsize=(10, 6))
cluster_summary[['Annual Income', 'Spending Score']].plot(kind='bar')
plt.title('Characteristics of Customer Segments')
plt.ylabel('Standardized Value')
plt.xticks(rotation=0)
plt.grid(True)
plt.show()
```

</details>

## 9. Summary and Key Takeaways

In this tutorial, we’ve covered:

1.  **Clustering algorithms** - K-means, hierarchical clustering, and
    DBSCAN
2.  **Cluster evaluation** - Determining the optimal number of clusters
    with the elbow method and silhouette analysis
3.  **Dimensionality reduction** - PCA and t-SNE for visualizing
    high-dimensional data
4.  **Feature loadings** - Interpreting principal components and their
    relationship to original features
5.  **Combined techniques** - Using dimensionality reduction and
    clustering together

### Next Steps

In the next tutorial, we’ll explore: - Text mining and natural language
processing - Sentiment analysis and topic modeling - Feature engineering
techniques

## 10. Additional Resources

-   [Scikit-learn Clustering
    Documentation](https://scikit-learn.org/stable/modules/clustering.html)
-   [Scikit-learn Dimensionality Reduction
    Documentation](https://scikit-learn.org/stable/modules/decomposition.html)
-   [An Introduction to Statistical
    Learning](https://www.statlearning.com/) - Chapter 10 covers
    unsupervised learning
-   [Principal Component Analysis Explained
    Visually](https://setosa.io/ev/principal-component-analysis/)
-   [t-SNE Explained Visually](https://distill.pub/2016/misread-tsne/)
-   [How to Use t-SNE
    Effectively](https://distill.pub/2016/misread-tsne/)
-   [A Visual Introduction to Machine Learning: Dimensionality
    Reduction](http://www.r2d3.us/visual-intro-to-machine-learning-part-2/)
-   [Clustering on the Iris
    Dataset](https://scikit-learn.org/stable/auto_examples/cluster/plot_cluster_iris.html)

## Appendix: Mathematical Foundations

### A.1 K-means Algorithm

The K-means algorithm works as follows:

1.  Initialize k cluster centers (randomly or using k-means++)
2.  Assign each data point to the nearest cluster center
3.  Update each cluster center as the mean of all points assigned to it
4.  Repeat steps 2-3 until convergence (centers no longer change
    significantly)

The objective is to minimize the within-cluster sum of squares:

$J = \sum_{i=1}^{k} \sum_{x \in C_i} \|x - \mu_i\|^2$

where $C_i$ is the set of points in cluster $i$ and $\mu_i$ is the
center of cluster $i$.

### A.2 Principal Component Analysis (PCA)

PCA finds a new coordinate system by:

1.  Centering the data (subtracting the mean)
2.  Computing the covariance matrix of the centered data
3.  Finding the eigenvectors and eigenvalues of the covariance matrix
4.  Sorting eigenvectors by their eigenvalues in descending order
5.  Projecting the data onto the top eigenvectors (principal components)

The first principal component captures the direction of maximum variance
in the data, the second captures the direction of second most variance
orthogonal to the first, and so on.

The explained variance ratio of component $i$ is:

$\text{Explained Variance Ratio}_i = \frac{\lambda_i}{\sum_{j=1}^{n} \lambda_j}$

where $\lambda_i$ is the eigenvalue corresponding to the $i$-th
eigenvector.

### A.3 Silhouette Score

The silhouette score for a single point is defined as:

$s(i) = \frac{b(i) - a(i)}{\max(a(i), b(i))}$

where: - $a(i)$ is the average distance from point $i$ to all other
points in the same cluster - $b(i)$ is the average distance from point
$i$ to all points in the nearest neighboring cluster

The silhouette score ranges from -1 to 1, where: - Values near 1
indicate the point is well-clustered - Values near 0 indicate the point
is on the border between clusters - Values near -1 indicate the point
might be assigned to the wrong cluster

The overall silhouette score is the average of all individual silhouette
scores.